# Diffusion 기반 손 관절 In-betweening (CondMDI 방식)

이 노트북은 팀의 SILK Transformer 백본을 **CondMDI**(Cohan et al., "Flexible Motion
In-betweening with Diffusion Models", SIGGRAPH 2024, arXiv:2405.11126,
공식 코드: github.com/setarehc/diffusion-motion-inbetweening) 방식의 diffusion
모델로 전환한 구현입니다.

## 왜 Diffusion인가
SILK(결정적 L1 회귀)는 시작·끝 컨텍스트가 비슷하면(예: 쥐었다 편 뒤 원래 모양으로 돌아옴)
"아무 일도 없었다"는 가장 안전한 답으로 수렴하는 경향이 있습니다(regression to the mean).
이건 문헌에서 "multimodal ambiguity"로 알려진 문제이며, CondMDI를 비롯한 diffusion 기반
in-betweening 연구들이 정확히 이 문제를 풀기 위해 나왔습니다 — 하나의 정답을 회귀하는 대신
가능한 중간 동작들의 분포에서 샘플링합니다.

## 팀 확정 사항 반영
- 데이터셋: **How2Sign만** (CSL-Daily 제외)
- 위치 인코딩: 팀이 검증한 **목표-상대(target-relative) 인코딩** 그대로 재사용
  (RMIB의 time-to-arrival 설계와 문헌적으로 일치함이 확인됨)
- **왼손 flip**: `diag(1,-1,-1)` conjugation, **행(row) 기반** 6D 컨벤션
  (SignSparK 공식 소스 `pose_datasets_lmdb.py`로 100% 확인됨)
- 평가: train(학습) / dev(개인 검증) / test(팀 최종 비교), L=[5,10,20,30]
- 지표: L2Q(부호보정 쿼터니언), L2P(실제 MANO로 근사, 팀원들의 손수 제작 템플릿보다 정밀),
  NPSS(FFT 기반)
- 전처리: 팀원 1이 만든 offset=5 stride 기반 인덱스(`train_index.npz` 등)를 그대로 사용.
  **단, 팀원 1의 인덱스 자체엔 flip이 적용된 실제 값이 없고 `hand` 플래그(0=오른손,
  1=왼손)만 있음 — 이 노트북이 실제 데이터 로딩 시점에 flip을 적용합니다.**
- 텍스트 조건화(translation/gloss): **보류** — 윈도우 단위로 텍스트 정렬이 안 맞고
  수어 전용 특화 위험이 있어 팀 논의로 제외.
- 다중 키프레임(segment 기반 보너스 조건): gap 안에 `segment` 상 수어 중간점이
  우연히 포함되면 그 프레임도 조건으로 노출 (CondMDI의 "무작위 개수 키프레임" 학습
  철학과 일치, 고정 3개가 아니라 "있으면 활용"하는 방식).

**공식 코드 확인 사항**: CondMDI는 MDM(Motion Diffusion Model) 위에 지어졌고, MDM 계열은
노이즈(ε)가 아니라 **x0(원본)를 직접 예측**하는 것이 특징입니다 — 이 노트북도 x0-prediction을
따릅니다. 학습 커맨드(`train.train_condmdi --keyframe_conditioned`)가 시사하듯, 핵심은
**"학습 때부터 마스킹된 조건(관측 프레임은 그대로, 나머지만 노이즈)"을 명시적으로 가르치는 것**
입니다 — 단순 추론시점 imputation은 논문에서 이미 성능이 떨어짐이 확인됐습니다.


## 1. 환경 설정

In [1]:

!pip install -q lmdb

import os, math, pickle, random, inspect, io
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from google.colab import drive
drive.mount('/content/drive')

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 344.7/344.7 kB 27.7 MB/s eta 0:00:00
Mounted at /content/drive
device: cuda


## 2. chumpy/MANO 설정 (L2P 평가용)

In [2]:

!pip install chumpy -q

if not hasattr(np, "bool"): np.bool = bool
if not hasattr(np, "int"): np.int = int
if not hasattr(np, "float"): np.float = float
if not hasattr(np, "object"): np.object = object
if not hasattr(np, "str"): np.str = str
if not hasattr(np, "complex"): np.complex = complex
if not hasattr(np, "unicode"): np.unicode = str
if not hasattr(inspect, "getargspec"): inspect.getargspec = inspect.getfullargspec

import chumpy
print("chumpy 로드 성공")

!pip install -q smplx
import smplx

MANO_MODEL_ROOT = "/content/drive/MyDrive/KUBIG/KUBIG SUMMER CONTEST"  # mano 폴더의 부모 경로

mano_right = smplx.create(
    model_path=MANO_MODEL_ROOT, model_type="mano", is_rhand=True,
    use_pca=False, flat_hand_mean=False, num_pca_comps=45,
).to(DEVICE)

print("MANO 로드 완료")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.6/50.6 kB 5.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


/tmp/ipykernel_1035/2763592422.py:6: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"): np.object = object
/tmp/ipykernel_1035/2763592422.py:7: FutureWarning: In the future `np.str` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "str"): np.str = str


chumpy 로드 성공
MANO 로드 완료


## 3. 설정값

관절/차원, context, 평가 T값, diffusion 스텝 수 등. 팀 확정 사항 그대로 반영.


In [3]:

N_JOINTS = 15          # How2Sign hand_pose (손목 제외)
POSE_DIM = N_JOINTS * 6  # 90

C_CONTEXT = 10
EVAL_T_VALUES = [5, 10, 20, 30]
TRAIN_T_RANGE = (5, 30)
MAX_SEQ_LEN = C_CONTEXT + max(TRAIN_T_RANGE[1], max(EVAL_T_VALUES)) + 1  # 41

D_MODEL = 1024
N_HEADS = 8
N_LAYERS = 6
D_FF = 4096
DROPOUT = 0.1

N_DIFFUSION_STEPS = 1000   # DDPM 표준 스텝 수
BATCH_SIZE = 64            # diffusion은 SILK보다 forward pass가 무거워서(매 스텝 노이즈 샘플링)
                            # 조금 더 작게 시작, OOM 나면 더 줄이기

INDEX_DIR = "/content/drive/MyDrive/KUBIG/KUBIG SUMMER CONTEST/SignSparK_index"  # 팀원1 인덱스 위치
DATA_ROOT = "/content/signspark_local_data"  # LMDB 로컬 경로 (기존과 동일 관례)


## 4. 왼손 flip + 6D 유틸리티

팀이 SignSparK 공식 소스(`pose_datasets_lmdb.py`)로 검증한 그대로: **행(row) 기반**
Gram-Schmidt 컨벤션, `diag(1,-1,-1)` conjugation.


In [4]:

_LEFT_HAND_FLIP_MASK = np.array([
    [ 1.0, -1.0, -1.0],
    [-1.0,  1.0,  1.0],
    [-1.0,  1.0,  1.0],
], dtype=np.float32)


def rotation_6d_to_matrix_np(d6):
    '''6D -> (...,3,3). 행(row) 기반 -- pose_datasets_lmdb.py와 동일 컨벤션.'''
    a1, a2 = d6[..., :3], d6[..., 3:]
    b1 = a1 / np.linalg.norm(a1, axis=-1, keepdims=True)
    b2 = a2 - np.sum(b1 * a2, axis=-1, keepdims=True) * b1
    b2 = b2 / np.linalg.norm(b2, axis=-1, keepdims=True)
    b3 = np.cross(b1, b2, axis=-1)
    return np.stack((b1, b2, b3), axis=-2)


def matrix_to_rotation_6d_np(mats):
    '''(...,3,3) -> 6D, 앞 2개 행.'''
    batch_dim = mats.shape[:-2]
    return mats[..., :2, :].copy().reshape(*batch_dim, 6)


def flip_left_hand_features(left_feats, n_joints=N_JOINTS):
    '''왼손(SMPLX-left) -> 오른손(WiLoR) 컨벤션으로 변환. 공식 로직과 동일.'''
    left = left_feats.astype(np.float32, copy=False)
    T_len = left.shape[0]
    pose_6d = left[:, :n_joints * 6].reshape(T_len * n_joints, 6)
    mats = rotation_6d_to_matrix_np(pose_6d) * _LEFT_HAND_FLIP_MASK
    flipped = matrix_to_rotation_6d_np(mats).reshape(T_len, n_joints * 6)
    if left.shape[-1] > n_joints * 6:
        return np.concatenate([flipped, left[:, n_joints * 6:]], axis=-1)
    return flipped


# ---- torch 버전 (모델/평가에서 사용, row 기준, pytorch3d 표준과 검증 완료) ----
def rotation_6d_to_matrix(d6):
    a1, a2 = d6[..., 0:3], d6[..., 3:6]
    b1 = torch.nn.functional.normalize(a1, dim=-1)
    b2 = a2 - (b1 * a2).sum(-1, keepdim=True) * b1
    b2 = torch.nn.functional.normalize(b2, dim=-1)
    b3 = torch.cross(b1, b2, dim=-1)
    return torch.stack([b1, b2, b3], dim=-2)


def matrix_to_axis_angle(R):
    batch_shape = R.shape[:-2]
    R_flat = R.reshape(-1, 3, 3)
    cos_theta = ((R_flat[:, 0, 0] + R_flat[:, 1, 1] + R_flat[:, 2, 2]) - 1) / 2
    cos_theta = cos_theta.clamp(-1 + 1e-7, 1 - 1e-7)
    theta = torch.acos(cos_theta)
    axis = torch.stack([
        R_flat[:, 2, 1] - R_flat[:, 1, 2],
        R_flat[:, 0, 2] - R_flat[:, 2, 0],
        R_flat[:, 1, 0] - R_flat[:, 0, 1],
    ], dim=-1)
    denom = (2 * torch.sin(theta)).clamp(min=1e-7).unsqueeze(-1)
    axis = axis / denom
    return (axis * theta.unsqueeze(-1)).reshape(*batch_shape, 3)


def sixd_sequence_to_axis_angle(seq_6d):
    if isinstance(seq_6d, np.ndarray):
        seq_6d = torch.from_numpy(seq_6d).float()
    return matrix_to_axis_angle(rotation_6d_to_matrix(seq_6d))

@torch.no_grad()
def mano_forward(mano_layer, hand_pose_aa, device=DEVICE):
    T = hand_pose_aa.shape[0]
    global_orient = torch.zeros(T, 3, device=device)
    hand_pose = hand_pose_aa.reshape(T, -1).to(device)
    betas = torch.zeros(T, 10, device=device)
    output = mano_layer(global_orient=global_orient, hand_pose=hand_pose, betas=betas, return_verts=True)
    return output.joints  # (T, 16, 3)


## 5. 팀 위치 인코딩 (목표-상대) 그대로 재사용

In [5]:

class RelativePositionalEncoding(nn.Module):
    '''목표 키프레임을 기준(상대위치 0)으로 한 학습 가능한 위치 임베딩.
    팀이 검증한 설계 그대로 -- RMIB의 time-to-arrival과 문헌적으로 일치.'''
    def __init__(self, d_model, max_len=200):
        super().__init__()
        self.max_len = max_len
        self.pos_embedding = nn.Embedding(2 * max_len + 1, d_model)

    def forward(self, x, rel_pos):
        idx = (rel_pos + self.max_len).clamp(0, 2 * self.max_len)
        return x + self.pos_embedding(idx)


## 6. Diffusion 유틸리티 — 노이즈 스케줄, timestep 임베딩

**Timestep 임베딩이란**: 학습은 정답에 노이즈를 단계적으로(t=1..T) 섞어가는 과정을 배우고,
추론은 순수 노이즈에서 거꾸로(T..1) 걷어내며 복원합니다. "노이즈가 살짝 낀 상태"와
"거의 완전한 노이즈 상태"는 모델이 해야 할 일이 다르므로, **"지금 몇 번째 단계인지(t)"를
위치 인코딩과 같은 방식(sinusoidal)으로 벡터화해서 입력에 더해줍니다.**

**노이즈 스케줄**: 표준 cosine schedule(Nichol & Dhariwal 2021 계열, 여러 최신 diffusion
연구에서 널리 쓰임)을 사용합니다.


In [6]:

def cosine_beta_schedule(timesteps, s=0.008):
    steps = timesteps + 1
    x = torch.linspace(0, timesteps, steps)
    alphas_cumprod = torch.cos(((x / timesteps) + s) / (1 + s) * math.pi * 0.5) ** 2
    alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
    betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
    return torch.clip(betas, 0.0001, 0.9999)


class DiffusionSchedule:
    def __init__(self, n_steps=N_DIFFUSION_STEPS, device=DEVICE):
        self.n_steps = n_steps
        betas = cosine_beta_schedule(n_steps).to(device)
        alphas = 1.0 - betas
        self.alphas_cumprod = torch.cumprod(alphas, dim=0)          # (n_steps,)
        self.sqrt_alphas_cumprod = torch.sqrt(self.alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - self.alphas_cumprod)

    def q_sample(self, x0, t, noise=None):
        '''x0: (B,T,D), t: (B,) int64 -> x_t'''
        if noise is None:
            noise = torch.randn_like(x0)
        sqrt_ac = self.sqrt_alphas_cumprod[t].view(-1, 1, 1)
        sqrt_om_ac = self.sqrt_one_minus_alphas_cumprod[t].view(-1, 1, 1)
        return sqrt_ac * x0 + sqrt_om_ac * noise, noise


class TimestepEmbedding(nn.Module):
    '''diffusion timestep t -> d_model 차원 벡터 (sinusoidal, 위치 인코딩과 동일한 수식).'''
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_model), nn.SiLU(), nn.Linear(d_model, d_model),
        )

    def forward(self, t):
        half = self.d_model // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device).float() / half)
        args = t.float()[:, None] * freqs[None]
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)  # (B, d_model)
        return self.mlp(emb)


## 7. 모델 — SILK 백본 + timestep 조건 + x0 예측

MDM/CondMDI 계열은 노이즈(ε)가 아니라 **x0(원본 회전값)을 직접 예측**합니다. 입력은
`[노이즈 낀 회전(90) ; 관측여부 플래그(1)]`이고, 관측된(컨텍스트+목표+보너스 키프레임)
구간은 노이즈를 안 섞고 **깨끗한 정답을 그대로** 넣습니다(CondMDI의 핵심 - "학습 때부터
inpainting 패턴을 가르친다").


In [7]:

class DiffusionSILKHand(nn.Module):
    def __init__(self, pose_dim=POSE_DIM, d_model=D_MODEL, n_heads=N_HEADS,
                 n_layers=N_LAYERS, d_ff=D_FF, dropout=DROPOUT, max_len=MAX_SEQ_LEN):
        super().__init__()
        self.input_proj = nn.Linear(pose_dim + 1, d_model)  # +1: 관측여부 플래그
        self.pos_enc = RelativePositionalEncoding(d_model, max_len=max_len)
        self.time_emb = TimestepEmbedding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True, activation="gelu", norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.output_proj = nn.Linear(d_model, pose_dim)

    def forward(self, x_noisy_with_flag, rel_pos, t, valid_mask=None):
        '''x_noisy_with_flag: (B, T, pose_dim+1) -- 관측 구간은 이미 깨끗한 값으로 치환됨
        rel_pos: (B, T) 목표-상대 위치
        t: (B,) diffusion timestep
        '''
        h = self.input_proj(x_noisy_with_flag)
        h = self.pos_enc(h, rel_pos)
        h = h + self.time_emb(t).unsqueeze(1)  # 모든 프레임에 동일한 timestep 정보 브로드캐스트
        key_padding_mask = ~valid_mask if valid_mask is not None else None
        h = self.encoder(h, src_key_padding_mask=key_padding_mask)
        return self.output_proj(h)  # x0 예측 (B, T, pose_dim)


In [8]:

import os, shutil

DRIVE_BACKUP = "/content/drive/MyDrive/KUBIG/KUBIG SUMMER CONTEST/SignSparK_lmdb_backup"


def setup_lmdb():
    os.makedirs(DATA_ROOT, exist_ok=True)
    drive_lmdb_dir = f"{DRIVE_BACKUP}/lmdb"
    local_lmdb_dir = f"{DATA_ROOT}/lmdb"

    def _all_present(base_dir):
        return all(
            os.path.exists(f"{base_dir}/{split}/How2Sign_reopt_{split}.lmdb/data.mdb")
            for split in ["train", "dev", "test"]
        )

    if _all_present(local_lmdb_dir):
        print("로컬에 이미 LMDB 있음, 그대로 사용")
        return

    if _all_present(drive_lmdb_dir):
        print("Drive 백업에서 로컬로 복사 중...")
        shutil.copytree(drive_lmdb_dir, local_lmdb_dir, dirs_exist_ok=True)
        print("복사 완료")
        return

    print("Drive 백업도 없음 -- 처음부터 다운로드 (한 번만, 이후엔 Drive 백업으로 재사용)")
    os.environ["HF_HUB_DISABLE_XET"] = "1"  # xet 백엔드 불안정 이력이 있어 비활성화
    if not os.path.exists("/content/SignSparK_repo"):
        os.system("git clone -q https://github.com/JianHe0628/SignSparK.git /content/SignSparK_repo")
    os.system(f'cd /content/SignSparK_repo && python tools/download_data.py '
              f'--datasets How2Sign --dest "{DATA_ROOT}"')

    print("Drive에 백업 중 (다음번엔 이 다운로드 단계를 생략할 수 있음)...")
    os.makedirs(DRIVE_BACKUP, exist_ok=True)
    shutil.copytree(local_lmdb_dir, drive_lmdb_dir, dirs_exist_ok=True)
    print("백업 완료")


setup_lmdb()

for split in ["train", "dev", "test"]:
    path = f"{DATA_ROOT}/lmdb/{split}/How2Sign_reopt_{split}.lmdb/data.mdb"
    exists = os.path.exists(path)
    size = f"({os.path.getsize(path)/1e9:.2f} GB)" if exists else ""
    print(f"{split}: {exists} {size}")

Drive 백업에서 로컬로 복사 중...
복사 완료
train: True (7.90 GB)
dev: True (0.45 GB)
test: True (0.59 GB)


## 8. 데이터 로더 — 팀원 1 인덱스 + 실제 LMDB + flip + segment

팀원 1의 `{split}_index.npz`(clip_idx, hand, start, [T])를 읽어 실제 프레임을 LMDB에서
가져오고, `hand==1`(왼손)이면 flip을 적용합니다. `segment`도 같이 가져와서, gap 구간 안에
수어 중간점이 있으면 보너스 키프레임으로 노출합니다.


In [9]:

import lmdb

def open_lmdb(split):
    path = f"{DATA_ROOT}/lmdb/{split}/How2Sign_reopt_{split}.lmdb"
    env = lmdb.open(path, readonly=True, lock=False, readahead=False, meminit=False, max_readers=1024)
    return env


def load_clip_raw(env, clip_id):
    with env.begin() as txn:
        raw = txn.get(clip_id.encode() if isinstance(clip_id, str) else clip_id)
    npz = np.load(io.BytesIO(raw), allow_pickle=True)
    return {
        "segment": npz["segment"],
        "left_features": npz["left_features"][:, :POSE_DIM],
        "right_features": npz["right_features"][:, :POSE_DIM],
    }


class DiffusionSignSparkDataset(Dataset):
    '''팀원1 인덱스 파일 기반. mode='train'이면 T를 매번 새로 뽑고(인덱스엔 start만 저장돼
    있음, SILK 논문 방식 그대로), mode='eval'이면 인덱스에 저장된 고정 T를 그대로 씀.'''

    def __init__(self, split, index_dir=INDEX_DIR, mode="train", seed=SEED):
        self.split = split
        self.mode = mode
        self.rng = np.random.default_rng(seed)

        idx_path = f"{index_dir}/{split}_index.npz"
        data = np.load(idx_path, allow_pickle=True)
        self.clip_ids = data["clip_ids"]
        self.clip_idx = data["clip_idx"]
        self.hand = data["hand"]
        self.start = data["start"]
        self.T_arr = data["T"] if "T" in data.files else None

        self.env = None  # lazy open (워커별로)
        self._clip_cache = {}  # 간단한 LRU 대용 -- 같은 클립이 여러 윈도우에 쓰이는 경우 재사용

    def _get_env(self):
        if self.env is None:
            self.env = open_lmdb(self.split)
        return self.env

    def _get_clip(self, clip_i):
        if clip_i not in self._clip_cache:
            if len(self._clip_cache) > 5000:  # 캐시 크기 제한
                self._clip_cache.clear()
            cid = self.clip_ids[clip_i]
            self._clip_cache[clip_i] = load_clip_raw(self._get_env(), cid)
        return self._clip_cache[clip_i]

    def __len__(self):
        return len(self.clip_idx)

    def __getitem__(self, i):
        clip_i = int(self.clip_idx[i])
        hand_flag = int(self.hand[i])
        start = int(self.start[i])
        clip = self._get_clip(clip_i)

        key = "left_features" if hand_flag == 1 else "right_features"
        feats = clip[key]
        if hand_flag == 1:
            feats = flip_left_hand_features(feats)

        if self.mode == "train":
            T = int(self.rng.integers(TRAIN_T_RANGE[0], TRAIN_T_RANGE[1] + 1))
        else:
            T = int(self.T_arr[i])

        L = C_CONTEXT + T + 1
        window = feats[start:start + L].astype(np.float32)          # (L, 90)
        segment_window = clip["segment"][start:start + L]            # (L,)

        target_rot = window.copy()
        obs_mask = np.zeros(L, dtype=bool)
        obs_mask[:C_CONTEXT] = True
        obs_mask[-1] = True

        # 보너스 키프레임: 학습 때만 노출 (평가/추론 때 노출하면 데이터 유출 -- 우리 태스크는
        # "gap 안의 정답을 전혀 모르는 상태에서 복원"하는 게 본질이라, eval에선 순수 컨텍스트+목표만 써야 함)
        if self.mode == "train":
            gap_range = slice(C_CONTEXT, L - 1)
            bonus_idx = np.where(segment_window[gap_range] == 2)[0] + C_CONTEXT
            obs_mask[bonus_idx] = True

        rel_pos = np.arange(L, dtype=np.int64) - (L - 1)  # 목표(마지막 프레임) 기준 상대위치

        return {
            "target": torch.from_numpy(target_rot),     # (L, 90) 정답
            "obs_mask": torch.from_numpy(obs_mask),      # (L,) True=관측(컨텍스트/목표/보너스)
            "rel_pos": torch.from_numpy(rel_pos),
        }


def collate_diffusion(batch):
    L_max = max(b["target"].shape[0] for b in batch)
    B = len(batch)
    target = torch.zeros(B, L_max, POSE_DIM)
    obs_mask = torch.zeros(B, L_max, dtype=torch.bool)
    valid_mask = torch.zeros(B, L_max, dtype=torch.bool)
    rel_pos = torch.full((B, L_max), -9999, dtype=torch.int64)

    for i, b in enumerate(batch):
        L = b["target"].shape[0]
        target[i, :L] = b["target"]
        obs_mask[i, :L] = b["obs_mask"]
        valid_mask[i, :L] = True
        rel_pos[i, :L] = b["rel_pos"]

    return {"target": target, "obs_mask": obs_mask, "valid_mask": valid_mask, "rel_pos": rel_pos}


## 9. 학습 — CondMDI 스타일 마스킹 + x0 예측 손실

매 스텝: 정답에 노이즈를 섞되, **관측 구간(컨텍스트/목표/보너스 키프레임)은 노이즈를
안 섞고 깨끗한 값 그대로 유지**합니다. 모델은 전체 시퀀스에 대해 x0를 예측하고, 손실은
SILK와 마찬가지로 **전체 시퀀스**에 대해 계산합니다(팀이 gap-only 손실 ablation은
이번 스코프에서 보류하기로 했으므로 일관성 유지).


In [10]:

def training_step(model, schedule, batch, device=DEVICE, use_amp=True):
    target = batch["target"].to(device)       # (B, L, 90) 정답
    obs_mask = batch["obs_mask"].to(device)     # (B, L) True=관측
    valid_mask = batch["valid_mask"].to(device)
    rel_pos = batch["rel_pos"].to(device)

    B, L, _ = target.shape
    t = torch.randint(0, schedule.n_steps, (B,), device=device)

    x_t, _ = schedule.q_sample(target, t)
    # 관측 구간은 노이즈 없이 깨끗한 값으로 강제 치환 (CondMDI 핵심 메커니즘)
    obs_mask_f = obs_mask.unsqueeze(-1).float()
    x_input = x_t * (1 - obs_mask_f) + target * obs_mask_f

    flag = obs_mask.float().unsqueeze(-1)  # 관측여부 플래그 채널
    model_in = torch.cat([x_input, flag], dim=-1)

    # bfloat16 사용 -- SILK/SignSparK 트랙에서 float16은 불안정(큰 유한값으로 발산),
    # bfloat16이 안전하다는 게 이미 검증됐음. 여기도 동일하게 적용.
    if use_amp and device.type == "cuda":
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            pred_x0 = model(model_in, rel_pos, t, valid_mask)
    else:
        pred_x0 = model(model_in, rel_pos, t, valid_mask)

    diff = torch.abs(pred_x0 - target).mean(dim=-1)  # (B, L)
    diff = diff * valid_mask.float()
    loss = diff.sum() / valid_mask.float().sum().clamp(min=1.0)
    return loss


def train_diffusion_silk(epochs=30, samples_per_epoch=None, lr=1e-4,
                          batch_size=BATCH_SIZE, warmup_steps=1000,
                          save_dir="/content/drive/MyDrive/KUBIG/KUBIG SUMMER CONTEST",
                          save_every=2, patience=5, resume=True):

    train_ds = DiffusionSignSparkDataset("train", mode="train")
    dev_ds = DiffusionSignSparkDataset("dev", mode="eval")

    if samples_per_epoch is None:
        samples_per_epoch = min(len(train_ds), 200_000)  # 전체 163만개는 epoch당 너무 크니 상한

    train_loader = DataLoader(
        torch.utils.data.Subset(train_ds, np.random.default_rng(0).choice(
            len(train_ds), samples_per_epoch, replace=False)),
        batch_size=batch_size, shuffle=True, collate_fn=collate_diffusion,
        num_workers=8,              # 1 -> 8 (12코어 중 메인 프로세스+여유분 남기고)
        pin_memory=True,
        persistent_workers=True,     # 추가
        prefetch_factor=4,           # 추가
    )
    dev_loader = DataLoader(dev_ds, batch_size=batch_size, shuffle=False,
                            collate_fn=collate_diffusion, num_workers=4,   # dev도 조금 올림
                            persistent_workers=True)

    model = DiffusionSILKHand().to(DEVICE)
    schedule = DiffusionSchedule()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    def lr_lambda(step):
        return min(1.0, step / max(warmup_steps, 1))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    latest_path = f"{save_dir}/diffusion_silk_latest.pt"
    best_path = f"{save_dir}/diffusion_silk_best.pt"
    start_epoch, best_val, epochs_no_improve = 0, float("inf"), 0

    if resume and os.path.exists(latest_path):
        ckpt = torch.load(latest_path, map_location=DEVICE)
        model.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optimizer_state"])
        start_epoch = ckpt["epoch"] + 1
        best_val = ckpt["best_val"]
        epochs_no_improve = ckpt["epochs_no_improve"]
        print(f"체크포인트에서 이어받음: epoch {start_epoch}부터, best_val={best_val:.4f}")

    for epoch in range(start_epoch, epochs):
        model.train()
        epoch_loss, n_batches = 0.0, 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
        for batch in pbar:
            optimizer.zero_grad()
            loss = training_step(model, schedule, batch)
            if torch.isnan(loss) or torch.isinf(loss):
                print("[경고] NaN/Inf loss, 배치 스킵")
                continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            epoch_loss += loss.item(); n_batches += 1
            pbar.set_postfix(loss=f"{epoch_loss/max(n_batches,1):.4f}")

        model.eval()
        val_loss, n_val = 0.0, 0
        with torch.no_grad():
            for batch in dev_loader:
                val_loss += training_step(model, schedule, batch).item()
                n_val += 1
        val_loss /= max(n_val, 1)
        print(f"[Epoch {epoch+1}] train={epoch_loss/max(n_batches,1):.4f} dev={val_loss:.4f}")

        torch.save({"model_state": model.state_dict(), "optimizer_state": optimizer.state_dict(),
                    "epoch": epoch, "best_val": best_val, "epochs_no_improve": epochs_no_improve},
                   latest_path)

        if val_loss < best_val - 1e-4:
            best_val = val_loss; epochs_no_improve = 0
            torch.save({"model_state": model.state_dict(), "epoch": epoch, "best_val": best_val},
                       best_path)
            print(f"  -> best 갱신, 저장됨")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print("Early stopping!"); break

    best_ckpt = torch.load(best_path, map_location=DEVICE)
    model.load_state_dict(best_ckpt["model_state"])
    return model, schedule


## 10. 추론(샘플링) — DDPM 역과정, 매 스텝 관측 구간 재치환

CondMDI 핵심: 노이즈 제거를 반복하는 매 스텝마다, 관측 구간(컨텍스트/목표/보너스)을
**그 스텝에 맞게 다시 노이즈를 살짝 섞은 정답값으로 재치환**합니다(순수 추론시점
imputation이 아니라, 학습 때와 동일한 절차를 추론에서도 반복 — 논문이 imputation
단독보다 이게 낫다고 보인 이유).

**⚠️ A100에서도 여기가 병목입니다.** 학습은 forward+backward 1회면 끝나지만, 샘플링은
`schedule.n_steps`(기본 1000)만큼 **순차적으로** forward를 반복해야 합니다(각 스텝이
이전 결과에 의존해서 병렬화 불가) — 배치 하나 채점에 forward 1000회가 필요한 셈이라,
test set 전체를 이 방식으로 평가하면 비현실적으로 오래 걸립니다.

**해결(DDIM 스타일 서브샘플링)**: diffusion 분야의 표준 기법으로, 1000 스텝으로 학습된
모델도 추론 때는 그 중 일부(`sampling_steps`, 예: 50)만 균일 간격으로 골라 밟아도 품질이
크게 떨어지지 않습니다. 아래 함수는 `sampling_steps` 인자로 이를 지원하고, SILK/SignSparK
트랙에서 이미 검증된 대로 float16 대신 **bfloat16**을 씁니다.


In [11]:

@torch.no_grad()
def sample_inbetween(model, schedule, target, obs_mask, rel_pos, valid_mask,
                      device=DEVICE, sampling_steps=50, use_amp=True):
    '''target: (B,L,90) -- 관측 구간의 정답값 소스로만 사용(gap 구간 값은 안 봄).
    sampling_steps: 실제로 밟을 스텝 수. schedule.n_steps(기본 1000)보다 작으면
    그 사이를 균일 간격으로 건너뛰며 진행 (DDIM 스타일 서브샘플링) -- A100에서도
    1000스텝 전체를 그대로 밟으면 평가가 비현실적으로 느려서 기본값을 50으로 낮춤.
    반환: (B,L,90) 최종 샘플.'''
    B, L, _ = target.shape
    x = torch.randn(B, L, POSE_DIM, device=device)

    sampling_steps = min(sampling_steps, schedule.n_steps)
    # 1000 스텝 중 sampling_steps개만 균일 간격으로 선택, 큰 t -> 작은 t 순으로 진행
    step_indices = torch.linspace(0, schedule.n_steps - 1, sampling_steps, device=device).long()
    step_indices = torch.unique(step_indices)  # 중복 제거(sampling_steps가 크면 중복 안 생김)
    step_indices = torch.flip(step_indices, dims=[0])

    for i in range(len(step_indices)):
        t_step = step_indices[i].item()
        t = torch.full((B,), t_step, device=device, dtype=torch.long)

        # 관측 구간을 이 시점(t)에 맞는 노이즈 낀 정답으로 재치환
        obs_mask_f = obs_mask.unsqueeze(-1).float()
        x_target_t, _ = schedule.q_sample(target, t)
        x_input = x * (1 - obs_mask_f) + x_target_t * obs_mask_f

        flag = obs_mask.float().unsqueeze(-1)
        model_in = torch.cat([x_input, flag], dim=-1)

        if use_amp and device.type == "cuda":
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                pred_x0 = model(model_in, rel_pos, t, valid_mask)
        else:
            pred_x0 = model(model_in, rel_pos, t, valid_mask)
        pred_x0 = pred_x0.float()  # 이후 스케줄 연산은 float32로 안전하게

        is_last = (i == len(step_indices) - 1)
        if not is_last:
            t_next = step_indices[i + 1].item()
            alpha_next = schedule.alphas_cumprod[t_next]
            noise = torch.randn_like(x)
            sqrt_ac_next = torch.sqrt(alpha_next)
            sqrt_om_ac_next = torch.sqrt(1 - alpha_next)
            x = sqrt_ac_next * pred_x0 + sqrt_om_ac_next * noise
        else:
            x = pred_x0

    # 마지막으로 관측 구간은 실제 정답으로 깔끔하게 덮어씀
    obs_mask_f = obs_mask.unsqueeze(-1).float()
    x = x * (1 - obs_mask_f) + target * obs_mask_f
    return x


## 11. 평가 지표 — L2Q, L2P(실제 MANO), NPSS

팀원 2가 만든 로직을 우리 row-기준 변환 + 실제 MANO로 재구현. 부호보정 쿼터니언 거리,
실제 forward kinematics, FFT 기반 리듬 유사도.


In [12]:

from scipy.spatial.transform import Rotation

def rotation_matrix_to_quaternion_np(mats):
    quats = Rotation.from_matrix(mats).as_quat()  # scipy: [x,y,z,w]
    return quats[:, [3, 0, 1, 2]]  # [w,x,y,z]로 재배열


def l2q_error(pred_90, gt_90, n_joints=N_JOINTS):
    T = pred_90.shape[0]
    pred_mats = rotation_6d_to_matrix_np(pred_90.reshape(T * n_joints, 6))
    gt_mats = rotation_6d_to_matrix_np(gt_90.reshape(T * n_joints, 6))
    pred_q = rotation_matrix_to_quaternion_np(pred_mats)
    gt_q = rotation_matrix_to_quaternion_np(gt_mats)
    dist = np.minimum(
        np.linalg.norm(pred_q - gt_q, axis=-1),
        np.linalg.norm(pred_q + gt_q, axis=-1),
    )
    return float(dist.mean())


def l2p_error_mano(pred_90, gt_90, mano_layer, n_joints=N_JOINTS):
    '''실제 MANO forward kinematics로 위치 오차 계산 (팀원들의 손수 제작 템플릿보다 정밀).'''
    T = pred_90.shape[0]
    pred_aa = sixd_sequence_to_axis_angle(pred_90.reshape(T, n_joints, 6))
    gt_aa = sixd_sequence_to_axis_angle(gt_90.reshape(T, n_joints, 6))
    pred_joints = mano_forward(mano_layer, pred_aa).cpu().numpy()
    gt_joints = mano_forward(mano_layer, gt_aa).cpu().numpy()
    return float(np.linalg.norm(pred_joints - gt_joints, axis=-1).mean())


def npss(pred_seq, gt_seq):
    pred_fft = np.abs(np.fft.fft(pred_seq, axis=0)) ** 2
    gt_fft = np.abs(np.fft.fft(gt_seq, axis=0)) ** 2
    gt_norm = gt_fft / (gt_fft.sum(axis=0, keepdims=True) + 1e-8)
    pred_norm = pred_fft / (pred_fft.sum(axis=0, keepdims=True) + 1e-8)
    diff = np.abs(gt_norm - pred_norm).sum(axis=0)
    weight = gt_fft.sum(axis=0)
    weight = weight / (weight.sum() + 1e-8)
    return float((diff * weight).sum())


def full_evaluate_diffusion(model, schedule, split="test", T_values=EVAL_T_VALUES,
                             mano_layer=mano_right, batch_size=16, max_batches_per_T=None,
                             sampling_steps=50):
    model.eval()
    results = {}
    ds = DiffusionSignSparkDataset(split, mode="eval")

    for T in T_values:
        T_indices = np.where(ds.T_arr == T)[0]
        sub = torch.utils.data.Subset(ds, T_indices)
        loader = DataLoader(sub, batch_size=batch_size, shuffle=False, collate_fn=collate_diffusion)

        l2q_list, l2p_list, npss_list = [], [], []
        for bi, batch in enumerate(tqdm(loader, desc=f"평가 T={T}", leave=False)):
            if max_batches_per_T is not None and bi >= max_batches_per_T:
                break
            target = batch["target"].to(DEVICE)
            obs_mask = batch["obs_mask"].to(DEVICE)
            valid_mask = batch["valid_mask"].to(DEVICE)
            rel_pos = batch["rel_pos"].to(DEVICE)

            pred = sample_inbetween(model, schedule, target, obs_mask, rel_pos, valid_mask,
                                     sampling_steps=sampling_steps)

            pred_np = pred.cpu().numpy()
            target_np = target.cpu().numpy()
            obs_np = obs_mask.cpu().numpy()
            valid_np = valid_mask.cpu().numpy()

            for b in range(pred_np.shape[0]):
                idx = np.where((~obs_np[b]) & valid_np[b])[0]  # gap 구간만 채점
                if len(idx) == 0:
                    continue
                gt_seg, pred_seg = target_np[b, idx], pred_np[b, idx]
                l2q_list.append(l2q_error(pred_seg, gt_seg))
                l2p_list.append(l2p_error_mano(pred_seg, gt_seg, mano_layer))
                npss_list.append(npss(pred_seg, gt_seg))

        results[T] = {"L2Q": np.mean(l2q_list), "L2P": np.mean(l2p_list), "NPSS": np.mean(npss_list),
                      "n": len(l2q_list)}
        print(f"T={T}: L2Q={results[T]['L2Q']:.4f} L2P={results[T]['L2P']:.4f} "
              f"NPSS={results[T]['NPSS']:.4f} (n={results[T]['n']})")

    return results


## 12. 실행

In [ ]:

model, schedule = train_diffusion_silk(epochs=30)


/tmp/ipykernel_2355/2744069172.py:13: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)


Epoch 1/30:   0%|          | 0/3125 [00:00<?, ?it/s]

[Epoch 1] train=0.0859 dev=0.0301
  -> best 갱신, 저장됨


Epoch 2/30:   0%|          | 0/3125 [00:00<?, ?it/s]

[Epoch 2] train=0.0319 dev=0.0247
  -> best 갱신, 저장됨


...

[Epoch 28] train=0.0196 dev=0.0164


Epoch 29/30:   0%|          | 0/3125 [00:00<?, ?it/s]

[Epoch 29] train=0.0194 dev=0.0163


Epoch 30/30:   0%|          | 0/3125 [00:00<?, ?it/s]

[Epoch 30] train=0.0195 dev=0.0164


In [13]:
model = DiffusionSILKHand().to(DEVICE)
ckpt = torch.load("/content/drive/MyDrive/KUBIG/KUBIG SUMMER CONTEST/diffusion_silk_best.pt", map_location=DEVICE)
model.load_state_dict(ckpt["model_state"])
model.eval()
schedule = DiffusionSchedule()  # 이건 학습 상태 없이 그냥 재생성하면 됨

print(f"불러온 체크포인트 epoch: {ckpt['epoch']}, best_val: {ckpt['best_val']:.4f}")

/tmp/ipykernel_1035/2744069172.py:13: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)


불러온 체크포인트 epoch: 26, best_val: 0.0164


In [14]:

test_results = full_evaluate_diffusion(model, schedule, split="test", batch_size=128, max_batches_per_T=30)


평가 T=5:   0%|          | 0/1155 [00:00<?, ?it/s]

T=5: L2Q=0.0258 L2P=0.0014 NPSS=0.0040 (n=3840)


평가 T=10:   0%|          | 0/1121 [00:00<?, ?it/s]

T=10: L2Q=0.0566 L2P=0.0031 NPSS=0.0232 (n=3840)


평가 T=20:   0%|          | 0/1053 [00:00<?, ?it/s]

T=20: L2Q=0.0919 L2P=0.0051 NPSS=0.0613 (n=3840)


평가 T=30:   0%|          | 0/986 [00:00<?, ?it/s]

T=30: L2Q=0.1178 L2P=0.0067 NPSS=0.0909 (n=3840)


In [15]:
from google.colab import runtime
runtime.unassign()